In [2]:
import os
import pandas as pd
import plotly.graph_objs as go
from plotly.subplots import make_subplots
from pathlib import Path

# Paths
cached_spectrums = Path("./cached_spectrums")

# Load precomputed medians from CSV
median_df = pd.read_csv(cached_spectrums / "median_snr_all.csv")

# Voltage types and their column names
voltage_types = [
    ("mininghighvoltage", "Mining High Voltage"),
    ("mininglowvoltage", "Mining Low Voltage"),
    ("soilhighvoltage", "Soil High Voltage"),
    ("soilmidvoltage", "Soil Mid Voltage"),
    ("soillowvoltage", "Soil Low Voltage")
]

# X-ray line catalog (keV)
LINE_CATALOG = {
    "elements": {
        "Na_Ka": 1.041, "Mg_Ka": 1.254, "Al_Ka": 1.486, "Si_Ka": 1.740,
        "P_Ka": 2.014, "S_Ka": 2.307, "Cl_Ka": 2.622, "K_Ka": 3.312,
        "Ca_Ka": 3.692, "Sc_Ka": 4.090, "Ti_Ka": 4.511, "V_Ka": 4.952,
        "Cr_Ka": 5.415, "Mn_Ka": 5.899, "Fe_Ka": 6.404, "Co_Ka": 6.930,
        "Ni_Ka": 7.478, "Cu_Ka": 8.048, "Zn_Ka": 8.638, "Ga_Ka": 9.251,
        "Ge_Ka": 9.886, "As_Ka": 10.543, "Se_Ka": 11.222, "Br_Ka": 11.924,
        "Rb_Ka": 13.396, "Sr_Ka": 14.165, "Y_Ka": 14.958, "Zr_Ka": 15.775,
        "Nb_Ka": 16.615, "Mo_Ka": 17.479, "Ru_Ka": 19.279, "Rh_Ka": 20.216,
        "Pd_Ka": 21.177, "Ag_Ka": 22.162, "Cd_Ka": 23.173,
        "Sn_Ka": 25.271, "Sb_Ka": 26.359, "Te_Ka": 27.472,
        "I_Ka": 28.612, "Ba_Ka": 32.194,
        "Rb_La": 1.839, "Sr_La": 1.806, "Y_La": 1.922,
        "Zr_La": 2.042, "Nb_La": 2.166, "Mo_La": 2.293,
        "Ru_La": 2.559, "Rh_La": 2.696, "Pd_La": 2.838,
        "Ag_La": 2.984, "Cd_La": 3.133, "In_La": 3.286,
        "Sn_La": 3.444, "Sb_La": 3.605, "Te_La": 3.770,
        "I_La": 3.938, "Ba_La": 4.466,
        "Pb_La": 10.551, "U_La": 13.613
    },
    "instrument": {
        "Rh_Ka": 20.216,
        "Rh_Kb": 22.724,
        "Rh_La": 2.696,
        "Ag_Ka": 22.162,
        "Ag_Kb": 24.942,
        "Ag_La": 2.984,
        "Rh_Rayleigh": 20.2,
        "Rh_Compton": 18.7,
        "Ag_Rayleigh": 22.2,
        "Ag_Compton": 20.5
    }
}

# Get all session names from the median file
session_names = median_df['session'].unique()

# Assign a unique color to each session for consistent legend/plot color
import plotly.colors as pc
palette = pc.qualitative.Plotly
color_map = {name: palette[i % len(palette)] for i, name in enumerate(session_names)}

# Plotly: 5 stacked plots, one per voltage
fig = make_subplots(rows=5, cols=1, shared_xaxes=True, vertical_spacing=0.03,
                    subplot_titles=[v[1] for v in voltage_types])

# For legend: only one entry per session (first voltage type)
legend_sessions = set()

# Add traces: for each session, add all voltages in the same order, and group bands with their median line using legendgroup. Only show legend for the first voltage for each session.
for j, folder_name in enumerate(session_names):
    color = color_map[folder_name]
    for i, (vcol, vname) in enumerate(voltage_types, 1):
        vdf = median_df[(median_df['session'] == folder_name) & (median_df['voltage'] == vcol)]
        if vdf.empty or not all(col in vdf.columns for col in ['q05','q25','q75','q95']):
            continue
        energy_axis = vdf['Energy (keV)'].values
        # 25-75 band
        x_band1 = list(energy_axis) + list(energy_axis[::-1])
        y_band1 = list(vdf['q25'].values) + list(vdf['q75'].values[::-1])
        # 5-95 band
        x_band2 = list(energy_axis) + list(energy_axis[::-1])
        y_band2 = list(vdf['q05'].values) + list(vdf['q95'].values[::-1])
        showlegend = False
        if folder_name not in legend_sessions:
            showlegend = True
            legend_sessions.add(folder_name)
        # Add bands (hidden from legend, but grouped with median line)
        fig.add_trace(
            go.Scatter(
                x=x_band2,
                y=y_band2,
                fill='toself',
                fillcolor='rgba(0,100,200,0.07)',
                line=dict(width=0),
                showlegend=False,
                name=f'{folder_name} Q05-Q95',
                hoverinfo='skip',
                legendgroup=folder_name,
            ),
            row=i, col=1
)
        fig.add_trace(
            go.Scatter(
                x=x_band1,
                y=y_band1,
                fill='toself',
                fillcolor='rgba(0,100,200,0.15)',
                line=dict(width=0),
                showlegend=False,
                name=f'{folder_name} Q25-Q75',
                hoverinfo='skip',
                legendgroup=folder_name,
            ),
            row=i, col=1
)
        # Median line (main legend entry)
        fig.add_trace(
            go.Scatter(
                x=energy_axis,
                y=vdf['SNR_median'],
                name=folder_name,
                mode='lines',
                legendgroup=folder_name,
                showlegend=showlegend,
                line=dict(color=color)
            ),
            row=i, col=1
)

# Add all X-ray lines as a single Scatter trace with hover labels, 1 SNR tall
for i, (vcol, vname) in enumerate(voltage_types, 1):
    x_lines = []
    y_lines = []
    text_lines = []
    for label, energy in list(LINE_CATALOG['elements'].items()) + list(LINE_CATALOG['instrument'].items()):
        x_lines.extend([energy, energy, None])
        y_lines.extend([0, 1, None])
        text_lines.extend([label, label, None])
    fig.add_trace(
        go.Scatter(
            x=x_lines,
            y=y_lines,
            mode='lines',
            line=dict(color='gray', dash='dot', width=1),
            showlegend=False,
            text=text_lines,
            hoverinfo='text'
)
        ,row=i, col=1
)

# Set keV ticks and labels on all plots, but only show x-axis title on the bottom plot
tickvals = list(range(0, int(max(median_df['Energy (keV)']))+5, 5))
ticktext = [str(x) for x in tickvals]
for i, (vcol, vname) in enumerate(voltage_types, 1):
    if i == len(voltage_types):
        # Only bottom plot gets the x-axis title
        fig.update_xaxes(
            title_text='Energy (keV)',
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            row=i, col=1,
            showticklabels=True,
            showline=True,
            mirror=True,
            ticks='outside',
            ticklen=8,
            tickwidth=2,
            tickcolor='black'
)
    else:
        fig.update_xaxes(
            title_text='',
            tickmode='array',
            tickvals=tickvals,
            ticktext=ticktext,
            row=i, col=1,
            showticklabels=True,
            showline=True,
            mirror=True,
            ticks='outside',
            ticklen=8,
            tickwidth=2,
            tickcolor='black'
)
    fig.update_yaxes(title_text="Median SNR", row=i, col=1)
fig.update_layout(
    height=1800,
    showlegend=True,
    title="Median SNR for Each Session (per Voltage)",
    legend_title_text="Session Folder",
)
fig.show(renderer="browser")
